### 0. Limpieza de matriz completa

In [1]:
## Lectura de librerías necesarias
library(readxl)
library(tidyverse)
library(igraph)
library(writexl)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Adjuntando el paquete: 'igraph'


The following objects are masked from 'package:lubridate':

    %--%, union


The following objects are masked from 'package:dplyr':

    as_data_frame, groups, union


The following objects are masked from 'package:purrr':

    compose, simplify


The following object is masked from 'package:tidyr':

    crossing


The following object is masked from 'package:tibble':

    as_data_frame


The following objects are masked from 'package:s

In [2]:
## Lectura de datos
path <- "C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Raw_Data\\Matriz_de_Adyacencia_Pensiones_SCOMP.xlsx"
df <- read_excel(path, sheet = "Hoja2", col_names = TRUE)

New names:
• `` -> `...1`


In [3]:
## Transformación a matriz de adyacencia
A <- df %>%
  rename(node = 1) %>%                            # primera columna = nombres fila
  mutate(across(-node, ~replace_na(as.numeric(.x), 0))) %>%  # NA->0, a numérico
  column_to_rownames("node") %>%
  as.matrix()

## Validación de la matriz de adyacencia
stopifnot(nrow(A) == ncol(A))
stopifnot(all(A %in% c(0,1)))
stopifnot(identical(rownames(A), colnames(A)))

# Autodependencias (i depende de sí mismo)
self_loops <- rownames(A)[diag(A) == 1]
self_loops

## Generación del grafo sin autodependencias
A_nodiag <- A
diag(A_nodiag) <- 0

g <- graph_from_adjacency_matrix(A_nodiag, mode = "directed", diag = FALSE)

c(vcount(g), ecount(g))  # (n_nodos, n_arcos)

## Dependencias mutuas (i depende de j y j depende de i)

mutual_idx <- which(A_nodiag == 1 & t(A_nodiag) == 1, arr.ind = TRUE)

direct_codep <- if (nrow(mutual_idx) == 0) {
  tibble(a = character(), b = character())
} else {
  pairs <- unique(t(apply(mutual_idx, 1, function(rc) sort(rc))))
  tibble(a = rownames(A_nodiag)[pairs[,1]],
         b = rownames(A_nodiag)[pairs[,2]])
}

direct_codep

# Verificación de que el grafo es un DAG (no tiene ciclos)
is_dag(g)

# Revisar componentes fuertemente conexas (ciclos)
scc <- components(g, mode = "strong")
groups <- split(V(g)$name, scc$membership)

problematic_scc <- groups[scc$csize > 1]
problematic_scc

character(0)

[1] 30 73

a,b
<chr>,<chr>


[1] TRUE

named list()

Ahora vamos a transformar esta matriz A a una forma que pueda ser leída en el modelo

In [4]:
nodes <- rownames(A)

concepts_base <- tibble(
  id   = seq_along(nodes),
  name = nodes
)

In [5]:
## Generación de la tabla de precedencias (i depende de j)
idx <- which(A == 1, arr.ind = TRUE)

precedence_names <- tibble(
  i = rownames(A)[idx[,1]],
  j = colnames(A)[idx[,2]]
)
## Mapeo de nombres a IDs usando el template de conceptos
lookup <- concepts_base %>% select(id, name)

precedence_tbl <- precedence_names %>%
  left_join(lookup, by = c("i" = "name")) %>% rename(i_id = id) %>%
  left_join(lookup, by = c("j" = "name")) %>% rename(j_id = id) %>%
  select(i = i_id, j = j_id)

#precedence_tbl

### 1. Construcción de pesos 

In [6]:
## creación de la tabla de métricas de importancia estructural para cada nodo
nodes <- rownames(A)
id_map <- tibble(id = seq_along(nodes), concept = nodes)

# Calcular métricas de importancia estructural para cada nodo

nodes <- V(g)$name

# Normalizar a [0,1]
norm01 <- function(x) (x - min(x)) / (max(x) - min(x) + 1e-9)


# Descendientes
D_out <- distances(g, v = nodes, to = nodes, mode = "out")
reach_out <- is.finite(D_out) & D_out > 0
desc <- rowSums(reach_out)

# Betweenness
btw <- betweenness(g, directed = TRUE, normalized = TRUE)

# Out-degree
outdeg <- degree(g, mode = "out")

# Nivel topológico = largo del camino más largo desde una raíz
topo <- as_ids(topo_sort(g, mode = "out"))
level <- setNames(rep(0, length(nodes)), nodes)

for (u in topo) {
  preds <- as_ids(neighbors(g, u, mode = "in"))
  level[u] <- if (length(preds) == 0) 0 else 1 + max(level[preds])
}

## Construir tabla con métricas normalizadas
w_tbl <- id_map %>%
  mutate(
    descendants  = desc[concept],
    outdeg       = outdeg[concept],
    betweenness  = btw[concept],
    level       = level[concept],
    descendants_n = norm01(descendants),
    outdeg_n      = norm01(outdeg),
    btw_n         = norm01(betweenness),
    foundation_n  = 1 - norm01(level)
  )

## Combinar métricas en una sola medida de importancia estructural

alpha <- 0.55
beta  <- 0
delta <- 0.15
gamma <- 0.3

w_tbl_fix <- w_tbl %>%
  mutate(
    w_raw = alpha*descendants_n + gamma*foundation_n + delta*btw_n + beta*outdeg_n
  )

# Ajuste para valores extremadamente bajos
pos <- w_tbl_fix$w_raw[w_tbl_fix$w_raw > 1e-12]
eps <- 0.05 * as.numeric(quantile(pos, 0.10))  # 5% del p10

w_tbl_fix <- w_tbl_fix %>%
  mutate(
    w_raw2 = ifelse(w_raw <= 1e-12, eps, w_raw),
    w = w_raw2 / sum(w_raw2),
    rank = rank(-w, ties.method = "min")
  )%>%
  select(id, concept, w)


### 2. Agregar tiempos

In [8]:
time_raw <- read_excel("C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Raw_Data\\estimacion_tiempo_lectura_36_contenidos (version 1).xlsb.xlsx", sheet = "Resultados", col_names = TRUE)

New names:
• `` -> `...6`
• `` -> `...9`
• `` -> `...12`
• `` -> `...14`


In [9]:
time_tbl <- time_raw %>%
  transmute(
    name_time = .[[2]],
    p_seconds = as.numeric(.[[9]])
  ) %>%
  filter(!is.na(name_time), name_time != "TOTAL")%>%
  mutate(key = name_time)

In [17]:
  concepts_tbl<-concepts_base %>%
  mutate(key = name) %>%
  left_join(time_tbl %>% select(key, p_seconds), by = "key") %>%
  left_join(w_tbl_fix %>% rename(name = concept), by = c("id", "name")) %>%
  mutate(
    # puedes cambiar a minutos si tu modelo lo requiere:
    # p = p_seconds / 60,
    p = p_seconds
  ) %>%
  select(id, name, p, w)

### Generar archivo final

In [11]:
# =========================
# 8) EXPORTAR EXCEL FINAL
# =========================
readme_tbl <- tibble(
  texto = c(
    "En hoja meta estan los parametros globales del modelo con las características propias de esta función.",
    "En hoja concepts estan los conceptos con id, nombre, tiempo p a partir de la estimación de 200 palabras por minuto y peso w con decendientes, out-degree, betweenness y nivel topológico (0.55 - 0 - 0.15 - 0.3 respectivamente).",
    "En hoja precedence estan las precedencias entre conceptos (i depende de j) de la matriz A completa."
  )
)

# Parametros meta (ajustar segun tu modelo)
meta_tbl <- tibble(
  parameter = c("h", "alpha", "beta", "v"),
  value     = c(150, 1, 0.3, 100)
)

In [18]:
write_xlsx(
  x = list(
    README     = readme_tbl,
    meta       = meta_tbl,
    concepts   = concepts_tbl,
    precedence = precedence_tbl
  ),
  path = "C:\\Users\\cfutr\\OneDrive\\Escritorio\\CICS\\2025\\Tesis\\Objetivos de Aprendizaje\\Modelo\\datos\\Intermedias\\instancia_modelo.xlsx")